In [ ]:
import sys
sys.path.append("..")

import torch
from torchviz import make_dot

from model.unet_upscaler_v1 import SuperResNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
checkpoint = torch.load('../checkpoints/upscaler.ckpt', map_location=DEVICE)
print(checkpoint['hyper_parameters'])

model = SuperResNet(start_channels=checkpoint['hyper_parameters']['start_channels'], 
                    upscale_block_length=2, 
                    downscale_block_length=2, 
                    depth=checkpoint['hyper_parameters']['depth']) 

state_dict = checkpoint['state_dict']
new_state_dict = {
    k.replace("model.", "").replace("_orig_mod.", ""): v 
    for k, v in state_dict.items()
}

model.load_state_dict(new_state_dict, strict=False)
model.to(DEVICE)
model.eval()

In [ ]:
from torchview import draw_graph

model.cpu() 
model.eval()

x = torch.randn(1, 3, 128, 128)

graph = draw_graph(
    model, 
    input_data=x,
    depth=1,            
    expand_nested=True,  
    save_graph=True,
    filename="block_architecture",
    device='cpu'
)

graph.visual_graph